# Does Γ's new gating tell a learned update from one that learned nothing?

*E46 stage 0, `experiments/regating_table/`, 2026-10-29. Record:
`reports/analyses/2026-10-29_analysis_e46-regating-table.md`. Plan: `reports/plans/2026-10-28_plan_regating-line-reset.md`.
Red-team review: `reports/reviews/2026-10-28_review_regating-line-red-team.md`.*

## The answer

Mostly yes, on toy tasks, with the random check still undecided. When the importance function is fed the fine-tuned
model's activations, the stand-in model's prediction of the answer rises by several units for every real fine-tune. Random
noise made as disruptive as each fine-tune moves it by at most an eighth as much, about as far as the noise moved the
real model's answer. That is too much to call nothing and too little to retire the line. A hand-made write that only
points the model at the answer lands close to the fine-tunes that teach a new mapping. The next step is the user's call.

**After Amendment 3.** The number stage 1 would carry, the stand-in ratio, was then read on the same nulls after
splitting them over the subcomponents. A hand-made write read about what the fine-tune of its own task read, low on one
new-mapping task and between the bars on the other. Random noise gave no reading at all. So the ratio sorts the tasks but does not
show on its own whether an update built anything. Whether to carry it to the larger model is the user's call.

## What we were trying to test

The user asked whether the change in Γ's values after a fine-tune "is genuinely describing how the model has changed, or
just some strange after-effect of it trying to predict a model it isnt trained on". A red-team review then showed that the
old headline share reads about the same for an update that learned nothing. Part of the reason is arithmetic: the share
divides by the real gain, which is near zero for such an update. So this stage drops the share and reads one number, in
nats, on one table, next to its nulls.

**The number.** Γ's values are computed twice on the same inputs, from the original model's activations and from the
fine-tuned model's. The **gate gain** is $g = L(C_0, G_1) - L(C_0, G_0)$:

- $L$, the CI-masked model's mean log-odds of the target at the scored position;
- $C_0$, the original subcomponents, so the update itself is left out;
- $G_0$ and $G_1$, Γ's values from the original and from the fine-tuned model.

A gate gain of 0 means Γ's new values give the old subcomponents nothing. The **real gain** $G = T_1 - T_0$ is the same
log-odds change in the full model, the yardstick. The gate gain lives in the masked stand-in only; it is never read in
the real fine-tuned model.

**The updates.** Seven toy fine-tunes of the 67M model, each a few Adam steps at two seeds, plus one that learned nothing.

- Three single-trigger tasks: after ` :)` predict `o`; after ` which` predict ` was`; after ` we` predict ` just`.
- Two copy tasks: after a marker and a key word, repeat the key, 32 keys per table. The model already copies, so this
  strengthens existing behaviour by construction.
- Two random-map tasks: the same keys, each mapped to a fixed random word. Nothing in the model does this, so it is new
  behaviour by construction.
- A fine-tune on code text that learned nothing: held-out text got 0.05 nats per token worse.

**The nulls.** Each is a forward pass on a changed copy of the original weights. Nothing is trained.

- *A fine-tune to another word*, ` http` after the same trigger: does any learned update of this size raise the target?
- *The fine-tune's update with its sign flipped*: does Γ follow the direction of the change?
- *A random update matched on side effects*: Gaussian, scaled until the full model's KL to the original on positions the
  task does not score equals the fine-tune's, at up to a hundred times the fine-tune's norm. A first run capped it at ten times, which stopped six of seven tasks short; the scored rows are the rerun. Does Γ react to a disturbance that
  points nowhere in particular?
- *A per-key write*: one change to block 1's MLP output matrix that maps each key's mean input onto its answer's output
  direction, scaled until its real gain matches the fine-tune's. The change itself lies outside the subcomponents, though its
  effect reaches the answer through the original later blocks. Does Γ hand the old
  subcomponents a gain whenever the activations start to predict the target?

The write is scored only on the random maps. On the copy tasks the answer is the key itself, and the model ties its input and
output embeddings, so the write there just adds the key's own embedding: an existing representation.

**The floor.** Per state, the larger of 0.1 nats and twice the standard error of the gate gain over held-out windows.

## What we expected, and why

- **The copy and random-map fine-tunes reproduce the earlier experiment.** Take what the updated subcomponents do
  unmasked. Under Γ's new values copy should keep at least 0.8 of it, and the random maps at most 0.5. This checks the
  rebuild; it is not new evidence.
- **The random update stays inside the floor.** An earlier size-matched random update was inert on emoticon. Matching side
  effects instead of size was meant to give it a fair chance. A quarter of the fine-tune's gate gain on any task would
  fail it; between the floor and that bar is inconclusive.
- **Nothing that lowers the target raises the gate gain.** The other-target fine-tunes and flipped updates should read at
  or below the floor.
- **The write reads well below the copy fine-tune.** If Γ simply turns on the answer's machinery whenever the activations
  predict the answer, the write would read like the copy fine-tune. At 0.8 of it or more the line is retired; at 0.5 or
  less it passes. We leaned medium.

## First, the control

The first control run stopped at once. The ` http` fine-tune lowered the emoticon target by about ten nats, and the filed
rule had asked for its real gain to stay near zero on both sides. That rule could never pass: learning ` http` after
` :)` must take probability from `o`. The user made the rule one-sided, so the null passes when it does not *raise* the
target. The rerun passed every check. The original model's readings matched the earlier experiments to within a hundredth
of a nat. The saved and rebuilt fine-tunes matched their filed gains within the 2% tolerance.

In [1]:
# Stage 0a, the control, from the committed results (no model).
import json
from pathlib import Path

R = Path("../experiments/regating_table/results")
a = json.loads((R / "stage0a.json").read_text())
print(f"{'group':22s} {'passed':>7s}")
for name, grp in a["groups"].items():
    print(f"{name:22s} {str(grp['ok']):>7s}")
print("whole control passed:", a["pass"])

group                   passed
e12/build                 True
e12/ident32               True
e12/ident32b              True
e12/rand32                True
e12/rand32b               True
e18/emoticon              True
e18/r1_t534_v369          True
e18/r2_t359_v816          True
e36/code                  True
whole control passed: True


## Does Γ's new gating help the old subcomponents for the fine-tunes but not for the nulls?

![gate gain of each fine-tune against each null on the same task](../experiments/regating_table/figures/gate_gain.png)

**Reading it.** Yes for the fine-tunes and for the nulls that lower the target; the random updates are the weak spot.
Every fine-tune, the blue circles, gives the old subcomponents 3.3 to 11.6 nats through Γ alone. Every update that
lowers the target reads a negative gate gain: the other-target fine-tunes in purple and the flipped updates in orange.
The random updates, grey squares, are matched to each fine-tune's side effects. Only one of them, on random map B, is
inside the shaded floor. The largest, on random map A, reads 0.48 nats, about half its red fail bar at 0.94. The others
sit further below their bars, and three of them are negative. The green triangles are the per-key writes, which land
between the nulls and the fine-tunes. The axis is stretched near zero, so small values look larger than they are.

In [2]:
# Gate gain per task in nats: seed-0 fine-tune and each null. Random rows from the KL-matched rerun.
b = json.loads((R / "stage0b.json").read_text())
S = b["states"]
M = json.loads((R / "amend1.json").read_text())
A = M["rows"]
TASKS = {"emoticon": "emoticon", "r1_t534_v369": "which", "r2_t359_v816": "we", "ident32": "copy A",
         "ident32b": "copy B", "rand32": "random map A", "rand32b": "random map B"}


def g(key):
    return f"{S[key]['g']:8.2f}" if key in S else f"{'':>8s}"


print(f"{'task':13s} {'fine-tune':>9s} {'random':>8s} {'flipped':>8s} {'write':>8s}")
for t, name in TASKS.items():
    print(f"{name:13s} {S[t + '/s0']['g']:9.2f} {A[t + '/random']['g']:8.2f} {g(t + '/flipped')} {g(t + '/write')}")

task          fine-tune   random  flipped    write
emoticon           8.02    -0.17    -2.76         
which              7.69    -0.11    -4.82         
we                 3.56    -0.11    -1.32         
copy A            11.23     0.21    -1.49     5.17
copy B            11.57     0.11    -1.68     6.71
random map A       3.75     0.48    -1.41     1.80
random map B       6.01     0.04    -0.86     3.92


## Did each random update reach the fine-tune's side effects?

![KL each random update reached against the fine-tune's, first run and rerun](../experiments/regating_table/figures/random_kl.png)

**Reading it.** Yes, on all seven tasks, once the scale was free to grow. The rule asks for a random update whose
off-target KL matches the fine-tune's within 10%. The first run capped the scale at ten times the fine-tune's norm, a
guard with no source, and six of seven rows stopped short: the hollow circles. Random map A reached only an eighth of its
target. A null weaker than its target does not run the rule as filed. So the cap was raised to a hundred times.
Every row then matched, the filled circles, at 6 to 26 times the norm. Before each row the rerun
recomputed the first run's update at the old scale and got the same KL exactly.

The same amendment also changed what a failed random check would lead to, which its own text did not say. Before, a
fail meant retiring the line; after, a fail goes to the user first, and a row that still cannot match gets its own
card. Both were written after the first run's results were seen, so a retire branch was softened after seeing data.
It did not change the outcome: the random check read inconclusive, not fail, and no row hit the new cap. The rerun's
prediction file still describes the cap as the only change; it stays as filed.


In [3]:
# Off-target KL of each random update against its target: first run at the cap, then the matched rerun.
print(f"{'task':13s} {'target':>7s} {'first':>7s} {'rerun':>7s} {'scale':>6s} {'same at old scale':>18s}")
for t, name in TASKS.items():
    old, new = S[t + "/random"], A[t + "/random"]
    print(f"{name:13s} {new['kl_target']:7.3f} {old['kl_other']:7.3f} {new['kl_other']:7.3f} {new['scale']:6.1f}"
          f" {str(new['repro']['ok']):>18s}")

task           target   first   rerun  scale  same at old scale
emoticon        0.055   0.044   0.051   10.8               True
which           0.122   0.055   0.124   14.8               True
we              0.090   0.046   0.084   13.4               True
copy A          0.189   0.070   0.192   16.1               True
copy B          0.205   0.072   0.195   16.1               True
random map A    0.696   0.085   0.716   26.4               True
random map B    0.097   0.107   0.103    6.4               True


## Does a random update's gate gain follow the real change it makes?

![gate gain against real gain for the matched random updates and the fine-tunes](../experiments/regating_table/figures/g_vs_G.png)

**Reading it.** Roughly, yes. A random update as disruptive as a fine-tune still moves the target a little in the real
model, by −0.20 to +0.40 nats. The grey squares lie near the dashed line where the gate gain equals that real change. On
the three tasks where the real change is over 0.1 nats, the gate gain is 0.79 to 1.21 times it. Where the real change is
near zero, the gate gain is within 0.12 of zero too; on ` which` the two have opposite signs. The fine-tunes, blue
circles, sit below the line at 0.27 to 0.80 of a real gain of 10 to 21 nats. So Γ passes on about as much of a random
update's small real change as of a fine-tune's large one. The filed rule scores only the gate gain's size, and by size
the random null reads inconclusive.

In [4]:
# The scored random rows: gate gain, real gain, floor and fail bar (a quarter of the seed-0 fine-tune's gate gain).
rows = M["score"]["P0.3"]["rows"]
print(f"{'task':13s} {'G':>7s} {'g':>7s} {'floor':>6s} {'fail bar':>9s}")
for t, name in TASKS.items():
    r = rows[t + "/random"]
    print(f"{name:13s} {r['G']:7.3f} {r['g']:7.3f} {r['F']:6.2f} {0.25 * abs(S[t + '/s0']['g']):9.2f}")
print("random null:", M["score"]["P0.3"]["status"], "| decision tree:", M["score"]["tree"])

task                G       g  floor  fail bar
emoticon       -0.196  -0.166   0.10      2.01
which           0.079  -0.111   0.10      1.92
we             -0.014  -0.105   0.10      0.89
copy A          0.264   0.209   0.12      2.81
copy B          0.037   0.108   0.10      2.89
random map A    0.401   0.485   0.21      0.94
random map B    0.051   0.044   0.10      1.50
random null: inconclusive | decision tree: card


## Does Γ treat a write that only points the activations at the answer like the copy fine-tune?

![per-key write's gate gain over the copy fine-tune's, with the pass and retire lines](../experiments/regating_table/figures/write_rho.png)

**Reading it.** No, but Γ does react to it. The scored writes, filled green triangles, read 0.16 and 0.34 of the copy
fine-tune's gate gain. Both are under the pass line at 0.5, so the retire test did not fire. The blue squares give the
proportion that matters: the real random-map fine-tunes read 0.31 to 0.52 of the copy fine-tune. The write sits below
the new-behaviour fine-tune, but not far below it. In nats it gave the old subcomponents 1.8 and 3.9, half and two-thirds of what the
random-map fine-tunes gave them. A write whose change lies outside the subcomponents still earns a gate gain far above the floor.
The hollow triangles are the copy writes, which add the key's own embedding. They read 0.46 and 0.58 and are not scored.

In [5]:
# The write test: gate gain over the seed-0 copy fine-tune's, for the same table size.
pairs = {"rand32": "ident32", "rand32b": "ident32b"}
print(f"{'table':6s} {'write':>6s} {'map ft s0':>10s} {'map ft s1':>10s} {'copy write':>11s}")
for rand, copy in pairs.items():
    ref = S[copy + "/s0"]["g"]
    vals = [S[k]["g"] / ref for k in (rand + "/write", rand + "/s0", rand + "/s1", copy + "/write")]
    print(f"{'A' if rand == 'rand32' else 'B':6s} {vals[0]:6.2f} {vals[1]:10.2f} {vals[2]:10.2f} {vals[3]:11.2f}")

table   write  map ft s0  map ft s1  copy write
A        0.16       0.33       0.31        0.46
B        0.34       0.52       0.50        0.58


## Do the copy and random-map fine-tunes reproduce the earlier experiment?

Yes, on all eight states, to the third decimal. Take what the updated subcomponents do unmasked. Under Γ's new values the
copy states keep 0.87 to 0.97 of it, and the random maps keep 0.36 to 0.41. This replicates the earlier result and adds no
new evidence: the gate gain alone already separates the classes, at about 11 nats for copy against 3.5 to 6 for the random maps.

## Amendment 3: does the stand-in ratio read the update's kind, or would a hand write read the same?

Stage 1 would carry one number to the larger model: the stand-in ratio. Give an update the least-squares split, the
least-norm change $(dU, dV)$ to the alive subcomponents with $dU V^\top + U dV^\top \approx \delta W$, the rest going to
Δ. In the stand-in model, rebuilt from the subcomponents alone, the subcomponent gain is how far the changed
subcomponents raise the answer's log-odds with every mask open. The kept gain is the same rise with each mask set to
Γ's importance value from the updated model. The ratio is kept gain over subcomponent gain: 1 means Γ switches on every
subcomponent that carries the change, 0 means none. It is undefined when the subcomponent gain is under 1 nat.

The question was whether an update with no learned machinery reads the same. So the per-key hand write and the random
update above, which had sat whole in Δ, were split exactly as the fine-tunes were and read by the same code. The filed rule
has two bars. Both random-map writes at or under 0.5 let stage 1 go ahead with this number. Either at or over 0.7 means
stage 1 needs a different number, and anything between goes to the user. The control passed: a zero update read
exactly 0, and every fine-tune and every null reproduced its earlier reading.

![stand-in ratio per table for the fine-tunes, the hand write and the undefined rows, against the 0.5 and 0.7 bars](../experiments/regating_table/figures/amend3.png)

**Reading it.** Roughly yes: on each table the hand write lands near the fine-tunes of that table, not near the other
kind. On random map table A the write read 0.40 against the fine-tunes' 0.36, under the low bar. On random map table B it
read 0.57 against 0.39 and 0.41, between the bars; its interval crosses the low bar. The copy writes read 0.89 and 1.13,
next to the copy fine-tunes. So the ratio keeps the copy tables apart from the random-map tables for fine-tunes and
writes alike, but it does not keep a write apart from a fine-tune of the same table. The crosses at the bottom are
undefined: random noise moves the subcomponents' gain by half a nat at most, so the ratio cannot be read on it. One
caveat: the copy writes carry the key's own embedding, so they are not a write free of existing machinery.


In [6]:
# Amendment 3: stand-in ratio per table after the least-squares split (undefined rows print as "-").
m = json.loads((R / "amend3.json").read_text())["rows"]
fmt = lambda r: f"{r['ratio']:.2f}" if r["defined"] else "-"
print(f"{'table':9s} {'ft s0':>6s} {'ft s1':>6s} {'write':>6s} {'random s0':>10s} {'write in-span':>14s}")
for t in ("ident32", "ident32b", "rand32", "rand32b"):
    r = {k: m[f"{t}/{k}"] for k in ("s0", "s1", "write_G", "random_s0")}
    print(f"{t:9s} {fmt(r['s0']):>6s} {fmt(r['s1']):>6s} {fmt(r['write_G']):>6s} {fmt(r['random_s0']):>10s}"
          f" {r['write_G']['in_span']:14.2f}")


table      ft s0  ft s1  write  random s0  write in-span
ident32     0.93   0.87   0.89          -           0.38
ident32b    0.93   0.97   1.13          -           0.38
rand32      0.36   0.36   0.40          -           0.39
rand32b     0.41   0.39   0.57          -           0.38


**What the branch means.** The filed tree ends at "card with the numbers". Neither random-map write reached the high
bar, so the ratio does not read a hand write like a copy fine-tune. But one write sat above the low bar, so stage 1 is
not proposed with this number as planned. Two more facts bear on it. The split subcomponents carry about half as much of a write as of a
fine-tune, 5 to 8 nats against 11 to 16. Only 38% of a write's energy lies in their span, against 80% for a fine-tune. And the random update,
the null stage 1 was to carry, gives no ratio on these toys at all. A candidate reading, not tested, is that the ratio
mostly reads where the task's answer sits among W₀'s subcomponents, not how the update reached it.

The method's standing limit applies here as everywhere on this line. VPD's unit is the rank-one subcomponent, not the
circuit, so a new mechanism built from subcomponents W₀ already has, under new importance values, reads as "elicited".
"Elicits", read from VPD, means "uses subcomponents W₀ already has", never "uses a circuit W₀ already runs".

E50, approved and next, reads the restore share in the real fine-tuned model on these same tables. This result leaves
its measurement and rule as they are. It weakens one of its follow-on branches: if E50 does not separate the two kinds,
stage 1 should go to the user rather than rest on the stand-in ratio alone.


## What we learned

Read in nats rather than as a share, the gate gain rises for updates that raise the target and stays at or below zero for
updates that lower it. That reframes the red team's sharpest complaint; it does not close it. Γ still passes on about the same
fraction of the real change, roughly four-fifths to one and a fifth, for the fine-tune that learned nothing and for the random
updates. A
disturbance as large as a fine-tune does not make Γ hand the old subcomponents gains of the fine-tune's order. It does
make Γ pass on most of whatever small real change it causes. The gate gain is also not a clean reading of "existing
machinery". A write whose change lies outside the subcomponents earns half to two-thirds of what a new-mapping fine-tune earns. `GOALS.md` asks
whether post-training strengthens an existing behaviour or teaches a new one. Here the copy fine-tunes stand apart from
both, but the line between "new behaviour" and "a change outside the subcomponents" is thin.

## How compelling is the evidence

- The control passed everywhere, and the earlier experiment's numbers were reproduced to the third decimal.
- The random null is the weak link. Matched on side effects on every task, six of its seven rows sit outside the floor.
  They are 2.2 to 4.8 standard errors from zero across held-out windows, not from the floor: one sits only 0.008 nats
  above it. The largest is half its fail bar; the rest are an eighth or less.
- Every null was drawn once per task: one random direction, one write, one flipped update. The floor is window noise for
  that one draw, not the spread between draws.
- Everything is on the 67M model with one scored position per task: three single-trigger tasks, two copy tables, two
  random-map tables and one code fine-tune.
- Every gate gain is read in the masked stand-in. Nothing here says what the real fine-tuned model does through these
  subcomponents.

## Surprises

- **Matching the random update pushed it out of the floor where we expected it inside.** Before the rerun we wrote down
  a pass on the single-trigger and copy tasks. All five instead read just outside the floor, and random map A's gate
  gain grew fourfold to 0.48 nats.
- **The random update's gate gain tracks its real gain.** Where the real gain is over 0.1 nats, the ratio is 0.79 to
  1.21, higher than any fine-tune's. Nothing was filed on this.
- **The per-key write earns a real gate gain.** It reads 1.8 and 3.9 nats, far above the floor. That is half to
  two-thirds of the random-map fine-tunes. No prediction compared it with them; the filed test used the copy fine-tune only.
- **The first run's cap bound on six tasks of seven.** The cap was filed as a guard, with no expected count.
- **After Amendment 3: no scale of a hand write raised the subcomponents' gain past about half a fine-tune's.** It peaked
  and then fell as the write grew. The scored prediction, both random-map writes at or under 0.5, missed on random map B.


## What next

The plan's decision tree sends an inconclusive random null beside a passed write test to the user as a decision. Three
things bear on it:

- **More random directions.** Several draws per task, matched as here. If the spread of their gate gains covers zero,
  the size question closes. If any reaches a quarter of the fine-tune's gate gain, the plan's retire branch applies.
- **A random update matched on the real gain.** If its gate gain still follows the real gain, Γ is passing on the
  change the update makes. That is what it should do; it would not be a reaction to the disturbance.
- **Whether the write is really a "no machinery" null.** The cosine between the write's and the fine-tune's change to the
  residual stream, proposed in review and not run. If it is high on the random maps, the write test cannot be read.
- **After Amendment 3: the stand-in ratio's role in stage 1.** The decision card is in
  `reports/analyses/2026-10-29_analysis_e46-amendment3.md`. E50, approved and next, reads the real-model restore share
  on the same tables. If it separates the two kinds, it carries stage 1 with the ratio beside it.
  If not, stage 1 goes back to the user rather than resting on the ratio alone.

## Scorecard

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| 0a | apparatus reproduces earlier runs | cells within 0.02; identity 0.597 ± 0.01; gains within 2%; ` http` null ≤ 0.1 of the gain, one-sided | all within; null −0.72 | pass |
| P0.2 | copy and random maps replicate E12 | gated piece gain over piece gain ≥ 0.8 copy, ≤ 0.5 random, both seeds | 0.87–0.97; 0.36–0.41 | pass |
| P0.3 | KL-matched random update moves no gate gain | all within F; any ≥ 0.25 × fine-tune g fails | all 7 matched; 1 of 7 within F; largest 0.485 against a bar of 0.94 | inconclusive |
| non-positive | G ≤ 0 never raises g | g ≤ 0.1 + F | every such row within | pass |
| P0.4 | random-map write ≠ copy fine-tune | ρ ≤ 0.5 pass, ≥ 0.8 retire | ρ = 0.16, 0.34 | pass |
| tree | | P0.2 pass, P0.3 inconclusive with no row capped, P0.4 pass | | card for the user |
| A3 control | split of a zero update reads 0; fine-tunes and nulls reproduce | both gains ≤ 0.001; ratios within 0.01; G, g within 2% | exact; 8 of 8; 8 of 8 | pass |
| A3.1 | split random-map writes do not read like copy | both ratios ≤ 0.5 | 0.40, 0.57 | fail on random map B |
| A3 tree | | one write in (0.5, 0.7) | | card with the numbers |

## Not run

- A second random direction per task, so the floor is window noise for one draw only.
- A random update matched on the real gain instead of the side effects.
- The per-key writes matched on side effects; they were scaled to the fine-tune's real gain only.
- A structured null of the fine-tune's size that is not the fine-tune.
- The residual-stream cosine between the write and the fine-tune.
- Writes at other layers or of other forms.
- The lowercase fine-tune, which sits at the floor; the teacher-training states, which no prediction reads.
- Any reading in the real fine-tuned model, any Γ from another seed or refitted, and anything on the larger model.
- After Amendment 3: more than one hand-write direction per table. Writes matched on side effects, or on the
  subcomponents' gain, which no scale reached. The split random update at the fine-tune's own norm. The ratio's KL form.

## Provenance

Control: Slurm 41161 stopped at the null; Slurm 42604 passed, 112 s, `results/stage0a.json`. Table: Slurm 42675, 109 s,
`results/stage0b.json`, log `cluster/logs/commute-regating-table-42675.out`. Matched random rows: Slurm 42791, about a
minute, `results/amend1.json`, log `cluster/logs/commute-regating-table-42791.out`. MATS cluster debug QoS, $0. Figures:
`experiments/regating_table/plot.py`. Amendment 3: Slurm 42993, 310 s, `results/amend3.json`, log `cluster/logs/commute-regating-table-42993.out`, figure
`experiments/regating_table/plot_amend3.py`; record `reports/analyses/2026-10-29_analysis_e46-amendment3.md`.